In [1]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [2]:

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except ValueError as e:
        print(f"Error mounting Google Drive: {e}. Please ensure you authorize access when prompted.")
    data_path = '/content/drive/MyDrive/f1-telemetry-ml'
else:
    data_path = '../fastf1_data/'

labeled_path = f'{data_path}/labeled'
processed_path = f'{data_path}/processed'

In [ ]:

target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']

train_labels = pd.read_parquet(f'{labeled_path}/train_data.parquet')
val_labels = pd.read_parquet(f'{labeled_path}/val_data.parquet')
test_in_dist_labels = pd.read_parquet(f'{labeled_path}/test_in_dist_data.parquet')
test_zero_shot_labels = pd.read_parquet(f'{labeled_path}/test_zero_shot.parquet')

segmented_files = glob.glob(f'{processed_path}/corners_*.parquet')
segmented = pd.concat([pd.read_parquet(f) for f in segmented_files], ignore_index=True)

SAFE_COLUMNS = ['Throttle', 'Brake', 'Speed', 'nGear', 'RPM']
merge_keys = ['year', 'race', 'session_type', 'driver', 'lap_number', 'corner_number']

def build_flat_features(labels_df, segmented_df):
    merged = segmented_df.merge(labels_df[merge_keys + target_cols], on=merge_keys, how='inner')
    rows = []
    for keys, group in merged.groupby(merge_keys):
        row = dict(zip(merge_keys, keys))
        for col in SAFE_COLUMNS:
            row[f'{col}_mean'] = group[col].mean()
            row[f'{col}_std'] = group[col].std()
            row[f'{col}_min'] = group[col].min()
            row[f'{col}_max'] = group[col].max()
        for t in target_cols:
            row[t] = group[t].iloc[0]
        rows.append(row)
    return pd.DataFrame(rows)

print("Building flat features (this takes a few minutes on the full dataset)...")
train_df = build_flat_features(train_labels, segmented)
val_df = build_flat_features(val_labels, segmented)
test_in_dist_df = build_flat_features(test_in_dist_labels, segmented)
test_zero_shot_df = build_flat_features(test_zero_shot_labels, segmented)

feature_cols = [c for c in train_df.columns if c.endswith(('_mean', '_std', '_min', '_max'))]
print(f"Train: {train_df.shape}, Val: {val_df.shape}")

model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
model.fit(train_df[feature_cols], train_df[target_cols])

def evaluate_split(df, split_name):
    preds = model.predict(df[feature_cols])
    mae = mean_absolute_error(df[target_cols], preds)
    rmse = np.sqrt(mean_squared_error(df[target_cols], preds))
    r2 = r2_score(df[target_cols], preds)
    print(f"\n--- Random Forest — {split_name} ---")
    print(f"Overall MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}")
    for i, col in enumerate(target_cols):
        col_mae = mean_absolute_error(df[col], preds[:, i])
        col_rmse = np.sqrt(mean_squared_error(df[col], preds[:, i]))
        col_r2 = r2_score(df[col], preds[:, i])
        print(f"  {col} — MAE: {col_mae:.4f}  RMSE: {col_rmse:.4f}  R2: {col_r2:.4f}")

evaluate_split(val_df, "Validation")
evaluate_split(test_in_dist_df, "Test (in-distribution)")
evaluate_split(test_zero_shot_df, "Test (zero-shot)")

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n=== Random Forest Feature Importance (top 10) ===")
print(importances.head(10))


Building flat features (this takes a few minutes on the full dataset)...
Train: (73848, 29), Val: (32350, 29)

--- Random Forest — Validation ---
Overall MAE: 0.0542  RMSE: 0.0789  R2: 0.8607
  aggression_score — MAE: 0.0383  RMSE: 0.0643  R2: 0.8155
  line_shape_score — MAE: 0.0517  RMSE: 0.0700  R2: 0.9280
  oversteer_preference_score — MAE: 0.0726  RMSE: 0.0982  R2: 0.8387

--- Random Forest — Test (in-distribution) ---
Overall MAE: 0.0346  RMSE: 0.0558  R2: 0.9218
  aggression_score — MAE: 0.0371  RMSE: 0.0653  R2: 0.8271
  line_shape_score — MAE: 0.0203  RMSE: 0.0295  R2: 0.9896
  oversteer_preference_score — MAE: 0.0465  RMSE: 0.0649  R2: 0.9488

--- Random Forest — Test (zero-shot) ---
Overall MAE: 0.0495  RMSE: 0.0849  R2: 0.9092
  aggression_score — MAE: 0.0356  RMSE: 0.0659  R2: 0.9083
  line_shape_score — MAE: 0.0402  RMSE: 0.0644  R2: 0.9488
  oversteer_preference_score — MAE: 0.0726  RMSE: 0.1147  R2: 0.8704

=== Random Forest Feature Importance (top 10) ===
Speed_min     

In [ ]:

if IN_COLAB:
    os.makedirs(f'{data_path}/models', exist_ok=True)
    joblib.dump(model, f'{data_path}/models/rf_model.pkl')
else:
    os.makedirs(f'../models', exist_ok=True)
    joblib.dump(model, f'../models/rf_model.pkl')
print("\nModel saved.")


Model saved.
